# コンピューティング

本章では、AWSソリューションアーキテクトアソシエイト試験に必要なコンピューティングサービスについて学習します。

## 本章の構成

- 仮想マシンサービス
- オートスケーリング
- バッチ処理
- サーバーレスコンピューティング
- ハイブリッドクラウド
- エッジコンピューティング

### 本章のサービス一覧

| | サービス | ひとこと |
|---|----------|----------|
| <img src="images/aws-icons/ec2.png" width="32" alt="Amazon EC2"/> | **Amazon EC2** | 仮想サーバー（VM） |
| <img src="images/aws-icons/autoscaling.png" width="32" alt="Amazon EC2 Auto Scaling"/> | **Amazon EC2 Auto Scaling** | 需要に応じてEC2台数を自動調整 |
| <img src="images/aws-icons/batch.png" width="32" alt="AWS Batch"/> | **AWS Batch** | 大量ジョブのバッチ実行 |
| <img src="images/aws-icons/lambda.png" width="32" alt="AWS Lambda"/> | **AWS Lambda** | イベント駆動のサーバーレス実行 |
| <img src="images/aws-icons/apigateway.png" width="32" alt="Amazon API Gateway"/> | **Amazon API Gateway** | HTTP/WebSocket APIの入口 |
| <img src="images/aws-icons/fargate.png" width="32" alt="AWS Fargate"/> | **AWS Fargate** | サーバーレスでコンテナ実行 |
| <img src="images/aws-icons/serverless-application-repository.png" width="32" alt="AWS Serverless Application Repository"/> | **AWS Serverless Application Repository** | サーバーレスアプリのテンプレートカタログ |
| <img src="images/aws-icons/outposts.png" width="32" alt="AWS Outposts"/> | **AWS Outposts** | オンプレにAWSインフラを設置 |
| <img src="images/aws-icons/vmware-cloud.png" width="32" alt="VMware Cloud on AWS"/> | **VMware Cloud on AWS** | vSphere環境をAWS上で運用 |
| <img src="images/aws-icons/wavelength.png" width="32" alt="AWS Wavelength"/> | **AWS Wavelength** | 5Gエッジで超低遅延コンピュート |

### 用語ミニ辞典（本章でよく出る語）

- **AMI（Amazon Machine Image）**：EC2起動用の仮想マシンの設計図
- **AZ（Availability Zone）**：リージョン内の独立したデータセンター群
- **フルマネージド**：AWSがパッチ適用や基盤運用を担うサービス形態

## 仮想マシンサービス

**この章で覚えること**

- **EC2**
- AMI・料金モデル（オンデマンド / RI / スポット）
- Placement Group で配置戦略

AWSの代表的な仮想マシンサービスは**Amazon EC2（Elastic Compute Cloud）**です。クラウド上に仮想サーバー（VM）を借りて使うイメージです。

- 幅広いインスタンスタイプ（**AMI** で起動）
- **セキュリティグループ**でネットワーク制御、永続ディスクは **EBS**

| インスタンスファミリー | 向く用途の例 |
|------------------------|--------------|
| 汎用（例：t3, m6i） | Webサーバー、開発環境 |
| コンピュート最適化（例：c6i） | 計算処理、バッチの一部 |
| メモリ最適化（例：r6i） | インメモリDB、大規模キャッシュ |

> **料金モデル（試験頻出）**
>
>

> | モデル | 特徴 | 向く例 |
> |--------|------|--------|
> | オンデマンド | 使った分だけ、縛りなし | 短期・検証 |
> | リザーブド / Savings Plans | 1〜3年のコミットで割引 | 安定稼働 |
> | スポット | 安いが中断されうる | 耐障害なバッチ |

- **向く例**：常時稼働するWeb/API、カスタムOSが必要なアプリ
- **向かない例**：サーバー管理を極力したくない短時間処理 → サーバーレス（Lambda）を検討

> **Placement Group（試験補足）**：インスタンスの配置戦略。**cluster**（低レイテンシ）、**spread**（障害ドメイン分散）、**partition**（大規模分散処理）。
---

## オートスケーリング

**この章で覚えること**

- **EC2 Auto Scaling** が台数を自動調整
- **ELB** とヘルスチェックで連携

オートスケーリングは、需要に応じて自動的にリソースの数を調整する技術です。AWSでは、**Amazon EC2 Auto Scaling**がこの機能を提供します。
EC2 Auto Scalingを利用することで、**トラフィックの増減に応じてEC2インスタンスの数を自動的に調整し、高可用性を実現**します。設定には、Auto Scalingグループ、ローンチテンプレート、スケーリングポリシーが含まれます。

Auto Scalingグループは、複数の**アベイラビリティゾーン（AZ）**にまたがってインスタンス数を管理します。最小数・最大数・希望数を設定し、ローンチテンプレートで起動設定を定義します。
スケーリングポリシーには、**ターゲット追跡**（例：CPU利用率を一定に保つ）や、スケジュールベースの方式があります。スケールイン時は**クールダウン**を設け、急な増減を防ぐのが一般的です。

- **向く例**：アクセス増減に応じてWebサーバー台数を自動調整
- **向かない例**：単一インスタンスのCPUサイズ変更のみ → EC2インスタンスタイプの変更を検討

**ELBとの連携**：ターゲットのヘルスチェック結果に応じてAuto Scalingがインスタンスを登録・解除します。

---

## バッチ処理

**この章で覚えること**

- **AWS Batch** で大量ジョブを一括実行
- コンピューティング環境に EC2 / スポットも利用可

バッチ処理は、大量のデータを一括して処理する方法です。AWSでは、**AWS Batch**がこれを提供します。
AWS Batchは、**バッチ処理ジョブを効率的に実行するサービスで、ジョブ定義、ジョブキュー、コンピューティング環境の設定を行い、自動的にリソースをプロビジョニングし、スケジューリング**します。

- **バッチ処理ジョブ**：一連のタスクを一括で処理するためのジョブ
- **ジョブ定義**：バッチジョブの設定を記述したもの
- **ジョブキュー**：バッチジョブを待機させるためのキュー
- **コンピューティング環境**：ジョブを実行するためのコンピューティングリソースの設定
- **プロビジョニング**：必要なリソースを自動的に割り当てるプロセス
- **スケジューリング**：ジョブを指定された時間や条件で実行すること

AWS Batchの主な使用例には、データ変換、ETL処理、シミュレーション、画像および動画処理があります。コンピューティング環境には**EC2**や**スポットインスタンス**を利用でき、コスト最適化に向きます。

- **向く例**：夜間にまとめて動かす大量バッチ、科学計算
- **向かない例**：リクエストごとの即時応答 → Lambda などを検討

---

## サーバーレスコンピューティング

**この章で覚えること**

- **Lambda**＋**API Gateway** が試験の定番
- **Fargate** はコンテナのサーバーレス実行

サーバーレスは、**OSやサーバー台数を意識せず、イベントに応じてコードを実行し、実行時間・回数に応じて課金**される形態です（裏でAWSがインフラを管理します）。

### AWS Lambda

**AWS Lambda**は、代表的なサーバーレス実行環境です。

関数をアップロードし、API Gateway・S3・SQS・SNS・EventBridgeなどのイベントで起動します。

### Amazon API Gateway

**Amazon API Gateway**

HTTP/WebSocket APIの公開、認証、スロットリング、LambdaやEC2バックエンドへのルーティングを担います。

サーバーレス構成の「入口」として試験で頻出です。

| 実行基盤 | サーバー管理 | 向く例 |
|----------|--------------|--------|
| **ECS on EC2** | あり | 既存コンテナ、細かいチューニング |
| **Fargate** | なし | コンテナをサーバーレス運用 |
| **Lambda** | なし | 短時間・イベント駆動 |

- **向く例**：APIの軽処理、ファイル変換、定期ジョブ、イベント連携
- **向かない例**：長時間の常時高負荷処理、低レイテンシの常時接続 → EC2 / ECS など

### AWS Fargate（補足）

コンテナをサーバーレスに動かす場合は **AWS Fargate**（ECS/EKS上）を使います。EC2を自分で管理せずにコンテナを稼働させたいときに選択肢になります。

### AWS Serverless Application Repository

公開されているサーバーレスアプリのテンプレートを検索・デプロイできるカタログです。Lambda構成の再利用に便利です。

**注意（試験で混同しやすい）**：**AWS Elastic Beanstalk**はPaaSであり、裏でEC2等が動くため**サーバーレスではありません**。コードをアップロードするとロードバランサーやスケーリングを自動構成してくれる「マネージドなアプリホスティング」です。

---

## ハイブリッドクラウド

**この章で覚えること**

- **Outposts**
- **VMware Cloud on AWS** でオンプレ統合

ハイブリッドクラウドは、オンプレミスとクラウドの両方のリソースを統合する環境です。AWSでは、**AWS Outposts**と**VMware Cloud on AWS**がこれに該当します。

AWS Outpostsは、オンプレミス環境にAWSインフラストラクチャとサービスを拡張するサービス。**低レイテンシが求められるアプリケーションやデータレジデンシ要件がある場合に利用されます**。
Outpostsは、完全管理型のハードウェアおよびソフトウェアスタックを提供し、AWSの同じツール、API、サービスをオンプレミスで使用できます。

VMware Cloud on AWSは、AWSインフラ上でVMware仮想化環境を実行するサービスです。**既存のVMwareツールを使用して、AWSリソースとシームレスに統合することができます**。

- **向く例**：オンプレとAWSで同じ運用ツールを使いたい、データを国内/社内に置く必要がある
- **向かない例**：クラウドのみで完結する新規システム

---

## エッジコンピューティング

**この章で覚えること**

- **Wavelength** が 5G エッジの超低遅延

エッジコンピューティングは、データの処理をデータ生成地点に近づけることで、遅延を最小限に抑える技術です。AWSでは、**AWS Wavelength**がこれを提供します。
AWS Wavelengthは、**5GネットワークのエッジでAWSサービスを提供するプラットフォーム**です。

AWS Wavelengthは、**超低レイテンシが求められるアプリケーションに最適であり、リアルタイム処理が求められるユースケースに適しています**。通信事業者の5Gネットワークに直接統合され、アプリケーションをエンドユーザーに近づけることで、遅延を最小限に抑えます。
具体的なユースケースとしては、AR/VR、リアルタイムゲーム、IoTデバイスのデータ処理などが挙げられます。

- **向く例**：5G端末近くで超低遅延が必要（AR/VR、リアルタイムゲーム）
- **向かない例**：一般的なWeb配信のみ → CloudFront などを先に検討

> ### 試験で押さえる設計の考え方（本章関連）
>
>

> - **可用性**：マルチAZ + Auto Scaling + ELB
> - **コスト**：RI/Savings Plans、スポット（Batch等）
> - **運用負荷削減**：Lambda / Fargate / API Gateway
## まとめ

- **EC2**：仮想サーバー。AMI・インスタンスタイプ・料金モデル・Placement Group
- **API Gateway + Lambda**：サーバーレスAPIの定番
- **ECS / Fargate**：コンテナ実行（EC2管理あり/なし）
- **EC2 Auto Scaling**：需要に応じた台数調整。マルチAZとターゲット追跡
- **AWS Batch**：大量ジョブの一括処理
- **Lambda / Fargate**：サーバーレス実行（BeanstalkはPaaSで別枠）
- **Outposts / VMware Cloud on AWS**：ハイブリッド統合
- **Wavelength**：5Gエッジでの超低遅延処理


## 復習クイズ

> **取り組み方**
> 1. 下の「問題」だけを読み、口頭またはメモで答える
> 2. すべて答えたあと、**区切りセル**まで進み、確認してから**解答・解説セル**へ進む
> 3. 解答が見えている場合は、紙などで隠してから取り組む

### 問題

**Q1.** EC2インスタンスを起動するときに使う、仮想マシンの設計図のことを何というか。

**Q2.** 1年間ほぼ一定の負荷でWebサーバーを稼働させ、コストを抑えたい。EC2の料金モデルとして適切なものは何か。

**Q3.** 夜間のみ、大量のETLジョブを並列で実行したい。どのサービスが最も適切か。

**Q4.** サーバー台数の管理をせず、S3へのアップロードをきっかけに短い処理を実行したい。サーバーレスとして適切なサービスは何か（Elastic Beanstalkは答えに含めないこと）。

**Q5.** アクセス増加に応じて、複数AZにまたがってEC2の台数を自動で増減させたい。使うべきサービスは何か。

**Q6.** 5Gネットワークのエッジで、AR/VR向けに超低遅延の処理を行いたい。AWSのサービス名は何か。

**Q7.** 外部向けREST APIを公開し、バックエンドのLambdaにルーティングしたい。入口となるサービス名は何か。


---

## ここまでで回答を考えてください

すべての問題（Q1〜Q7）に、口頭またはメモで答えてから、**次のセル**へ進んでください。

解答・解説は次のセルにあります。まだ答えを考えていない場合は、下にスクロールしないでください。

---


## 復習クイズ　解答・解説

**A1.** **AMI（Amazon Machine Image）**  
EC2起動時のOS・設定のテンプレート。→ 仮想マシンサービス

**A2.** **リザーブドインスタンス** または **Savings Plans**  
長期・安定稼働向けの割引モデル。オンデマンドは縛りなしだが割引は小さい。→ 仮想マシンサービス（料金モデル）

**A3.** **AWS Batch**  
ジョブ定義・キュー・コンピューティング環境で一括バッチ処理。→ バッチ処理  
※よくある誤答：Lambda（短時間・イベント単位向け）

**A4.** **AWS Lambda**  
イベント駆動でコード実行。BeanstalkはPaaS（裏でEC2）でありサーバーレスではない。→ サーバーレスコンピューティング

**A5.** **Amazon EC2 Auto Scaling**  
Auto Scalingグループとスケーリングポリシーで台数を調整。→ オートスケーリング

**A6.** **AWS Wavelength**  
5Gエッジでの超低遅延処理。一般的なWeb配信のみならCloudFront等を検討。→ エッジコンピューティング

**A7.** **Amazon API Gateway**  
HTTP APIの公開・認証・スロットリング。Lambdaと組み合わせる構成が一般的。→ サーバーレスコンピューティング
